# Webinar 2: Data Preprocessing — Track 3: Image Pipeline
This notebook covers the computer vision preprocessing and dimensionality reduction workflow:
1. **Loading**: Reading multi-class images from directory structures and ensuring RGB channel consistency.
2. **Resize**: Aspect-ratio preserving letterboxing vs direct resize.
3. **Normalize**: Pixel intensity scaling $[0, 255] \to [0.0, 1.0]$ and channel-wise standardization.
4. **PCA**: Principal Component Analysis for dimensionality reduction (>99% feature compression).
5. **Reduced Features & Reconstruction**: Scree plots, variance retention, and image reconstruction from principal components.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from src.image import (
    load_image_dataset,
    batch_resize_images,
    images_to_numpy,
    normalize_minmax,
    flatten_images,
    ImagePCA,
    run_image_pipeline
)

print('Image preprocessing modules loaded!')

## Step 1: Loading & Visualizing Raw Images

In [ ]:
raw_images, labels, file_paths, class_to_idx = load_image_dataset('../data/image/raw')

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, ax in enumerate(axes):
    ax.imshow(raw_images[i])
    ax.set_title(f'{labels[i]}\nSize: {raw_images[i].size}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Step 2: Standardizing Image Resolution (Letterbox vs Direct)

In [ ]:
resized_images = batch_resize_images(raw_images, target_size=(64, 64), preserve_aspect=True)
print(f'Standardized size: {resized_images[0].size}')

## Step 3: Pixel Intensity Normalization [0.0, 1.0]

In [ ]:
img_tensor = images_to_numpy(resized_images)
norm_tensor = normalize_minmax(img_tensor)
print(f'Raw min/max: {img_tensor.min():.1f}, {img_tensor.max():.1f}')
print(f'Normalized min/max: {norm_tensor.min():.1f}, {norm_tensor.max():.1f}')

## Step 4: PCA Dimensionality Reduction & Scree Plot
Reduce 12,288 flattened pixel dimensions to principal components retaining 95% variance.

In [ ]:
X_flat = flatten_images(norm_tensor)
pca_engine = ImagePCA(n_components=0.95)
X_pca = pca_engine.fit_transform(X_flat)
print(f'Original dimensions: {X_flat.shape[1]}')
print(f'Reduced PCA dimensions: {X_pca.shape[1]} ({(1 - X_pca.shape[1]/X_flat.shape[1])*100:.1f}% reduction)')
pca_engine.get_variance_summary(top_k=8)

## Step 5: Image Reconstruction from PCA & Model Evaluation

In [ ]:
image_results = run_image_pipeline('../data/image/raw')

from src.evaluation.visualizer import plot_image_pca_and_reconstruction
plot_image_pca_and_reconstruction(image_results, output_path='../reports/image_pca_and_reconstruction.png')

from src.evaluation.comparison import generate_modality_comparison
comp_df = generate_modality_comparison(image_results['baseline_metrics'], image_results['preprocessed_metrics'], 'Image')
comp_df